# Train the segmentation model on real labels (M3)

Runs on a Colab or Kaggle GPU (Runtime -> Change runtime type -> GPU). Each experiment config
in `ml/configs/train/` trains once per seed, is evaluated on the frozen real test split with
95 % bootstrap CIs, and is exported to ONNX for the edge benchmarks.

In [ ]:
!git clone --depth 1 https://github.com/justasillyboy/autonomous_drone.git
%cd autonomous_drone
!pip install -q ./uav_core ./edge ./ml ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data: the 286 labelled real frames from the asset release

In [ ]:
!scripts/fetch_assets.sh dataset-real-286.zip
!mkdir -p data/real && unzip -q -o assets/dataset-real-286.zip -d data/real
!uav-ml-split --dataset data/real --verify ml/configs/splits/real-286.json

## Experiments

E1 (label budget), then E3 (model size) and E4 (mask ratio). Three seeds each: plan about
1 GPU-hour per config on a T4.

In [ ]:
import subprocess

CONFIGS = ["e1_real_25", "e1_real_50", "e1_real_100", "e3_yolo11s", "e3_yolov8n", "e4_mask_ratio2"]
SPLIT = "ml/configs/splits/real-286.json"
for name in CONFIGS:
    cmd = ["uav-ml-train", f"ml/configs/train/{name}.yaml", "--dataset", "data/real"]
    subprocess.run([*cmd, "--runs", "runs", "--split", SPLIT], check=True)

## Collect results (summary CSVs, test reports, ONNX exports)

In [ ]:
import glob

import pandas as pd
from google.colab import files

summary = pd.concat(pd.read_csv(p) for p in glob.glob("runs/*/summary.csv"))
display(summary.groupby(["experiment"]).agg(["mean", "std"]).round(3))
!zip -q -r results.zip runs/*/summary.csv runs/*/seed*/eval_test.json runs/*/seed*/weights/best.onnx
files.download("results.zip")